In [122]:
import pandas as pd
import json
import sqlite3
from pyproj import Transformer

In [123]:
connection = sqlite3.connect("data/all_hiski_records.sqlite3")
cursor = connection.cursor()

query = """
    SELECT id, name, lon, lat
    FROM parish;
"""

cursor.execute(query)
toponym_to_coord_fi = {
    row[1]: {
        "lon": row[2],
        "lat": row[3],
    }
    for row in cursor.fetchall()
}

In [124]:
with open("outputs/json/db_toponyms_with_coordinates.json", "r") as fp:
    normalization_dict = json.load(fp)

## Checking for toponyms that won't get coordinates from database parish table in any case

In [125]:
normalization_dict_vals = pd.DataFrame.from_records(list(normalization_dict.values())).drop_duplicates()
toponyms_missing_coords = normalization_dict_vals[normalization_dict_vals["e"].isna()]["normalized"].to_list()

Lets remove this "(Ruotsi)" part:

In [127]:
list(
    filter(lambda s: "(" in s, toponyms_missing_coords)
)

['Alatornio (Ruotsi)',
 'Karunki (Ruotsi)',
 'Tornio (Ruotsi)',
 'Ylitornio (Ruotsi)']

In [128]:
toponyms_missing_coords = list(map(lambda x: x.replace(" (Ruotsi)", ""), toponyms_missing_coords))

Doesn't cause duplicates:

In [129]:
len(toponyms_missing_coords) == len(set(toponyms_missing_coords))

True

Let's also remove this "muuttokirja":

In [130]:
list(
    filter(lambda s: "muuttokirja" in s, toponyms_missing_coords)
)

['muuttokirja takaisin']

In [131]:
toponyms_missing_coords.remove("muuttokirja takaisin")

## Checking for missing toponyms from normalization json

In [132]:
display(set(toponym_to_coord_fi.keys()).difference(
    set(normalization_dict.keys())
))

print("Missing:", len(
    set(toponym_to_coord_fi.keys()).difference(
        set(normalization_dict.keys())
    )
))

{''}

Missing: 1


## Getting coordinates

Two sources for coordinates:

- https://www.maanmittauslaitos.fi/kartat-ja-paikkatieto/aineistot-ja-rajapinnat/tuotekuvaukset/nimisto
- https://www.geonames.org/ 

Prioritising Nimistö, use geonames for rest and if resolve places still left over manually.

In [133]:
nimisto = pd.read_json("data/json/municipalities_from_nimisto.jsonl", lines=True)
nimisto_coords = nimisto.set_index("spelling").to_dict(orient="index")

In [134]:
geonames_connection = sqlite3.connect("data/geonames_data/FI/FI.sqlite3")
geonames_cursor = geonames_connection.cursor()

In [135]:
geonames_cols = {c[0]: c[1] for c in geonames_cursor.execute(
    "PRAGMA table_info(FI);"
).fetchall()}

In [136]:
transformer = Transformer.from_crs("WGS84", "ETRS-TM35FIN").transform

In [137]:
geonames = pd.DataFrame(
    geonames_cursor.execute(
        "SELECT * FROM FI WHERE feature_class='A'"
    ).fetchall()
).rename(columns=geonames_cols)

geonames_filtered = geonames.loc[
    ~geonames.sort_values("modification_date", ascending=False).duplicated(
        subset="name", keep="first"
    )
].sort_values("name")

geonames_filtered = geonames_filtered[["name", "alternate_names", "latitude", "longitude"]]
geonames_coords = geonames_filtered.set_index("name").to_dict("index")
geonames_coords = {
    k: dict(
        zip(
            ("e", "n"),
            transformer(v["latitude"], v["longitude"])
        )
    )
    for k, v in geonames_coords.items()
}

In [139]:
from collections import Counter

import numpy as np
import rapidfuzz

for m in toponyms_missing_coords:
    #if m in nimisto_coords:
        #c.update(("nimistö",))
    #elif m in geonames_coords:
        #c.update(("geonames",))
    #else:
        #c.update(("not found",))
    nimisto_match = rapidfuzz.process.extractOne(
        m,
        nimisto_coords.keys(),
        scorer=rapidfuzz.distance.Levenshtein.normalized_similarity
    )
    geonames_match = rapidfuzz.process.extractOne(
        m,
        geonames_coords.keys(),
        scorer=rapidfuzz.distance.Levenshtein.normalized_similarity
    )
    
    if nimisto_match[1] >= 0.75 or geonames_match[1] >= 0.75:
        best = np.argmax((nimisto_match[1], geonames_match[1]))
        print(m, "->", (nimisto_match, geonames_match)[best], ("nimistö", "geonames")[best])

Alatornio -> ('Alatornio', 1.0, 4) geonames
Hamina -> ('Hamina', 1.0, 263) nimistö
Heinola -> ('Heinola', 1.0, 134) nimistö
Joutsa -> ('Joutsa', 1.0, 167) nimistö
Jyväskylä -> ('Jyväskylä', 1.0, 49) nimistö
Karunki -> ('Karunki', 1.0, 114) geonames
Metsämaa -> ('Metsämaa', 1.0, 233) geonames
Savonlinna -> ('Savonlinna', 1.0, 268) nimistö
Pohjanmaa -> ('Pohjanmaa', 1.0, 299) geonames
Tornio -> ('Tornio', 1.0, 251) nimistö
Ylitornio -> ('Ylitornio', 1.0, 92) nimistö


In [140]:
with open("outputs/json/db_spelling_groups.json", "r") as fp:
    spelling_groups = json.load(fp)
with open("outputs/json/db_spelling_to_group_id.json", "r") as fp:
    spelling_to_group_id = json.load(fp)

In [142]:
for t in toponyms_missing_coords:
    for s in spelling_groups[
        str(spelling_to_group_id[t])
    ]:
        geonames_match = rapidfuzz.process.extractOne(
            s,
            geonames_coords.keys(),
            scorer=rapidfuzz.distance.Levenshtein.normalized_similarity
        )
    
        good_match = geonames_match[1] >= 0.75
        if good_match:
            print(">>> ", end="")
        print(s, "->", geonames_match, "geonames")
    print("---")

Ensimmäinen Suomen meriekipaasi -> ('Keski-Suomen Lääni', 0.32258064516129037, 134) geonames
1:sta Finska Sjö Equipaget -> ('Eastern Finland Province', 0.2692307692307693, 15) geonames
1.Finska SjöEquip. -> ('Honkajoki', 0.2777777777777778, 48) geonames
Finska SjöEquipag -> ('Honkajoki', 0.2941176470588235, 48) geonames
1. F. Sjö-Equipaget -> ('Savitaipale', 0.26315789473684215, 362) geonames
1. Suomen meriekipaasi -> ('Suomenniemi', 0.40909090909090906, 394) geonames
1. Finska Sjö- Equip. -> ('Helsinki sub-region', 0.2857142857142857, 43) geonames
1. F: Sjö Equipaget. -> ('Savitaipale', 0.25, 362) geonames
1 sta Finska Sjö Equipag -> ('Eastern Finland Province', 0.25, 15) geonames
Första finska sjöekipaget -> ('Jakobstadsregionen', 0.28, 72) geonames
---
3. Oulun ruotujakoinen tarkk'ampujapataljoona -> ('Pielisen Karjala', 0.24444444444444446, 293) geonames
---
5. Mikkelin ruotujakoinen tarkk'ampujapataljoona -> ('Pielisen Karjala', 0.27083333333333337, 293) geonames
---
7. Porin ruot

In [35]:
geonames[geonames["name"].str.startswith("Karinainen")]

,geo_name_id,name,ascii_name,alternate_names,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,population,elevation,dem,timezone,modification_date
325,654060,Karinainen,Karinainen,"Karinainen,Karinais,Karinoinen",60.66667,22.83333,A,ADMD,FI,,15,,,,0,,74,Europe/Helsinki,2012-01-18


In [81]:
toponyms_missing_coords

['Ensimmäinen Suomen meriekipaasi',
 "3. Oulun ruotujakoinen tarkk'ampujapataljoona",
 "5. Mikkelin ruotujakoinen tarkk'ampujapataljoona",
 "7. Porin ruotujakoinen tarkk'ampujapataljoona",
 "9. Viipurin ruotujakoinen tarkk'ampujapataljoona",
 'Ahvenanmaa',
 'Alatornio',
 'Bruuhala',
 'Leineperi',
 'Haaparanta',
 'Hamiltonin rykmentti',
 'Hamina',
 'Heinola',
 'Hietaniemi',
 'Joutsa',
 'Jyväskylä',
 'Jägerhornin rykmentti',
 'Karunki',
 'Kerimäki Waara by',
 'Kuusjoki',
 'Kyrö',
 'Lappvesi',
 'Messukylä',
 'Metsämaa',
 'Muurla',
 'Neder Calix',
 'Norrtälje',
 'Savonlinna',
 'Pohjanmaa',
 'Ruotsi',
 'Suomen kaaderiekipaasi',
 'Suomen kaarti',
 'Suomen kadettikoulu',
 "Suomen tarkk'ampujapataljoona",
 'Tornio',
 'Uskela Pertteli',
 'Venäjä',
 'Västerås',
 'Ylikannus',
 'Ylitornio']